In [ ]:
import requests
from bs4 import BeautifulSoup
import json

In [ ]:
def get_category_links(base_url):
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"}
    response = requests.get(base_url, headers=headers)
    if response.status_code != 200:
        print(f"Failed to retrieve {base_url}")
        return []

    soup = BeautifulSoup(response.text, "html.parser")
    category_links = [a.get("href") for a in soup.select(".mntl-header-nav__sublist-item a") if a.get("href")]
    return category_links

# Base URL for categories
base_url = "https://www.allrecipes.com/recipes/"
category_urls = get_category_links(base_url)


In [ ]:
def get_recipe_links(category_url, max_pages=5):
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"}
    recipe_links = []

    for page in range(1, max_pages + 1):
        url = f"{category_url}?page={page}"
        response = requests.get(url, headers=headers)
        if response.status_code != 200:
            print(f"Failed to retrieve {url}")
            continue

        soup = BeautifulSoup(response.text, "html.parser")
        links = soup.select("a[href^='https://www.allrecipes.com/recipe/']")
        for link in links:
            href = link.get("href")
            if href and href.startswith("https://www.allrecipes.com/recipe/"):
                recipe_links.append(href)
    print(f"Extracting recipes from {url}")
    print(f"Found {len(recipe_links)} recipes")

    return list(set(recipe_links))  # Remove duplicates

all_recipe_urls = []
for category in category_urls:
    recipe_urls = get_recipe_links(category, max_pages=5)
    all_recipe_urls.extend(recipe_urls)

Extracting recipes from https://www.allrecipes.com/recipes/17057/everyday-cooking/more-meal-ideas/5-ingredients/main-dishes/?page=5
Found 295 recipes
Extracting recipes from https://www.allrecipes.com/recipes/15436/everyday-cooking/one-pot-meals/?page=5
Found 135 recipes
Extracting recipes from https://www.allrecipes.com/recipes/1947/everyday-cooking/quick-and-easy/?page=5
Found 20 recipes
Extracting recipes from https://www.allrecipes.com/recipes/455/everyday-cooking/more-meal-ideas/30-minute-meals/?page=5
Found 315 recipes
Extracting recipes from https://www.allrecipes.com/recipes/17889/everyday-cooking/family-friendly/family-dinners/?page=5
Found 220 recipes
Extracting recipes from https://www.allrecipes.com/recipes/94/soups-stews-and-chili/?page=5
Found 20 recipes
Extracting recipes from https://www.allrecipes.com/recipes/16099/everyday-cooking/comfort-food/?page=5
Found 160 recipes
Extracting recipes from https://www.allrecipes.com/recipes/80/main-dish/?page=5
Found 35 recipes
Ext

In [ ]:
def scrape_recipe_ingredients(url):
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"}
    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        print(f"Failed to retrieve {url}")
        return None

    soup = BeautifulSoup(response.text, "html.parser")

    # Extract JSON-LD script
    script_tag = soup.find("script", type="application/ld+json")
    if script_tag:
        data = json.loads(script_tag.string)

        # If it's a list, get the first element
        if isinstance(data, list):
            data = data[0]

        title = data.get("name", "No title")
        ingredients = data.get("recipeIngredient", [])

        return {
            "title": title,
            "ingredients": ingredients
        }

    print(f"Could not find JSON data for {url}")
    return None


recipe_ingredients = []
for url in all_recipe_urls:
    recipe = scrape_recipe_ingredients(url)
    if recipe:
        recipe_ingredients.append(recipe)

# Save extracted ingredients to a JSON file
if recipe_ingredients:
    with open("recipes_ingredients.json", "w", encoding="utf-8") as f:
        json.dump(recipe_ingredients, f, indent=4)
    print(f"{len(recipe_ingredients)} recipes with ingredients saved successfully!")


1248 recipes with ingredients saved successfully!
